# Wolfson — Training Notebook

Trains the Wolfson jazz phrase model on the Weimar Jazz Database.

**Before you start:**
- Set runtime to GPU: Runtime → Change runtime type → T4 GPU
- Download `wjazzd.db` from [jazzomat.hfm-weimar.de](https://jazzomat.hfm-weimar.de/download/download.html)
  and place it at `MyDrive/wolfson/wjazzd.db`, **or** use the direct upload cell (2b)

**Steps:**
1. Clone repo and install dependencies
2. Mount Google Drive and copy `wjazzd.db`
3. Inspect the database
4. Prepare training data (notes + chord sequences)
5. Train
6. Save model to Drive / download

## 1. Clone repo and install dependencies

In [ ]:
!git clone https://github.com/davidderoure/wolfson.git
%cd wolfson
!pip install -q pretty_midi numpy torch

## 2. Mount Google Drive and copy wjazzd.db

Place `wjazzd.db` at `MyDrive/wolfson/wjazzd.db` before running this cell.

In [ ]:
from google.colab import drive
import shutil, os

drive.mount('/content/drive')

DRIVE_WOLFSON = '/content/drive/MyDrive/wolfson'
os.makedirs(DRIVE_WOLFSON, exist_ok=True)
os.makedirs('data/raw', exist_ok=True)

db_src = f'{DRIVE_WOLFSON}/wjazzd.db'
db_dst = 'data/raw/wjazzd.db'

if os.path.exists(db_src):
    shutil.copy(db_src, db_dst)
    print(f'Copied wjazzd.db ({os.path.getsize(db_dst)/1e6:.1f} MB)')
else:
    print(f'Not found at {db_src} — use the upload cell below instead.')

## 2b. (Alternative) Upload wjazzd.db directly

Skip if you used Google Drive above.

In [ ]:
# from google.colab import files
# import os
# os.makedirs('data/raw', exist_ok=True)
# uploaded = files.upload()   # select wjazzd.db
# for fname in uploaded:
#     with open(f'data/raw/{fname}', 'wb') as f:
#         f.write(uploaded[fname])
#     print(f'Saved {fname} to data/raw/')

## 3. Inspect the database

In [ ]:
!python data/prepare.py --inspect

## 4. Prepare training data

Extracts saxophone phrases and chord sequences from the WJD.
Outputs:
- `data/processed/sax_sequences.npy` — interleaved pitch+duration token sequences
- `data/processed/sax_chords.npy` — chord index per token position
- `data/processed/sax_meta.json` — per-phrase metadata

In [ ]:
!python data/prepare.py --instrument sax

## 5. Check GPU and train

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(torch.cuda.get_device_name(0))

In [ ]:
# Adjust as needed. 100 epochs takes ~15–20 min on a T4.
INSTRUMENT = 'sax'
EPOCHS     = 100
BATCH_SIZE = 64

!python generator/train.py \
    --instrument {INSTRUMENT} \
    --epochs     {EPOCHS} \
    --batch-size {BATCH_SIZE}

## 6. Save trained model

In [ ]:
import shutil, os

INSTRUMENT = 'sax'
src = f'models/{INSTRUMENT}_best.pt'

if os.path.exists(src):
    # Save to Drive
    drive_dst = f'{DRIVE_WOLFSON}/{INSTRUMENT}_best.pt'
    shutil.copy(src, drive_dst)
    print(f'Saved to Drive: {drive_dst}')
    # Also offer direct download
    from google.colab import files
    files.download(src)
else:
    print(f'Model not found at {src} — check training completed.')

## 7. (Optional) Resume training from a checkpoint

In [ ]:
# INSTRUMENT = 'sax'
# checkpoint_on_drive = f'{DRIVE_WOLFSON}/{INSTRUMENT}_best.pt'
#
# import shutil, os
# os.makedirs('models', exist_ok=True)
# shutil.copy(checkpoint_on_drive, f'models/{INSTRUMENT}_best.pt')
#
# !python generator/train.py \
#     --instrument {INSTRUMENT} \
#     --epochs     50 \
#     --batch-size 64 \
#     --resume     models/{INSTRUMENT}_best.pt

## 8. (Optional) Train trumpet model

The WJD contains 102 trumpet + 15 cornet solos.

In [ ]:
# !python data/prepare.py --instrument trumpet
# !python generator/train.py --instrument trumpet --epochs 100 --batch-size 64